# Matmul: PMPP/Ch. 3

Single-thread and thread-per-element kernels written CUDA and Triton + benchmarking

In [ ]:
import cupy as cp
import numpy as np
import torch
import triton
import triton.language as tl
import time

print(f'GPU: {cp.cuda.Device()}')

WIDTH = 256
# input is the identity matrix
M_cp = cp.eye(WIDTH, dtype=cp.float32)
N_cp = cp.eye(WIDTH, dtype=cp.float32)
M_t  = torch.eye(WIDTH, device='cuda', dtype=torch.float32)
N_t  = torch.eye(WIDTH, device='cuda', dtype=torch.float32)

Giga nested loop inside of **one** thread. There's no parallelism at all.

#### Single-thread (CUDA)



In [ ]:
matmul_single_cuda = cp.RawKernel(r'''
extern "C" __global__
void matmul_single_cuda(const float *M, const float *N, float *P, int width) {
    for (int i = 0; i < width; i++)
        for (int j = 0; j < width; j++) {
            float sum = 0.0f;
            for (int k = 0; k < width; k++)
                sum += M[i * width + k] * N[k * width + j];
            P[i * width + j] = sum;
        }
}
''', 'matmul_single_cuda')

P = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
matmul_single_cuda(
    (1, 1), (1, 1),
    (M_cp.ravel(), N_cp.ravel(), P.ravel(), np.int32(WIDTH))
)
assert cp.allclose(P, cp.eye(WIDTH)), 'WRONG'
print('Single-thread CUDA: correct')

#### Single-thread (Triton)

In [ ]:
@triton.jit
def matmul_single_triton(M_ptr, N_ptr, P_ptr, width):
    for i in range(width):
        for j in range(width):
            acc = 0.0
            for k in range(width):
                a = tl.load(M_ptr + i * width + k)
                b = tl.load(N_ptr + k * width + j)
                acc += a * b
            tl.store(P_ptr + i * width + j, acc)

P_t = torch.zeros((WIDTH, WIDTH), device='cuda', dtype=torch.float32)
matmul_single_triton[(1,)](M_t, N_t, P_t, WIDTH)  # grid of 1 program

assert torch.allclose(P_t, torch.eye(WIDTH, device='cuda')), 'WRONG'
print('Single-thread Triton: correct')

![image.png](../imgs/ch3mm.png)
Both outer loops disappear. All of these iterations run once in parallel. Each thread now computes one output element of P and only the inner loop stays.


(Outer loops don't disappear but are now the grid demension)

#### Thread-per-element (CUDA)

In [ ]:
matmul_parallel_cuda = cp.RawKernel(r'''
extern "C" __global__
void matmul_parallel_cuda(const float *M, const float *N, float *P, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < width && col < width) {
        float sum = 0.0f;
        for (int k = 0; k < width; k++)
            sum += M[row * width + k] * N[k * width + col];
        P[row * width + col] = sum;
    }
}
''', 'matmul_parallel_cuda')

BLOCK = 16
GRID  = WIDTH // BLOCK

P2 = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
matmul_parallel_cuda(
    (GRID, GRID), (BLOCK, BLOCK),
    (M_cp.ravel(), N_cp.ravel(), P2.ravel(), np.int32(WIDTH))
)
assert cp.allclose(P2, cp.eye(WIDTH)), 'WRONG'
print(f'Thread-per-elem CUDA: correct')
print(f'{GRID}x{GRID} blocks x {BLOCK}x{BLOCK} threads = {WIDTH*WIDTH:,} threads launched')

#### Thread-per-element (Triton)

In [ ]:
@triton.jit
def matmul_parallel_triton(M_ptr, N_ptr, P_ptr, width, BLOCK_SIZE: tl.constexpr):
    # tl.program_id(0/1) == blockIdx.x/y in CUDA
    row = tl.program_id(1) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)[:, None]
    col = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)[None, :]

    acc = tl.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=tl.float32)

    # Inner k loop — same as the CUDA thread-per-elem kernel
    for k in range(0, width, BLOCK_SIZE):
        k_idx = k + tl.arange(0, BLOCK_SIZE)

        m = tl.load(M_ptr + row * width + k_idx[None, :],
                    mask=(row < width) & (k_idx[None, :] < width), other=0.0)
        n = tl.load(N_ptr + k_idx[:, None] * width + col,
                    mask=(k_idx[:, None] < width) & (col < width), other=0.0)

        acc += tl.dot(m, n)

    tl.store(P_ptr + row * width + col, acc,
             mask=(row < width) & (col < width))


BLOCK_SIZE = 16
GRID_T = WIDTH // BLOCK_SIZE

P2_t = torch.zeros((WIDTH, WIDTH), device='cuda', dtype=torch.float32)
matmul_parallel_triton[(GRID_T, GRID_T)](
    M_t, N_t, P2_t, WIDTH, BLOCK_SIZE=BLOCK_SIZE
)

assert torch.allclose(P2_t, torch.eye(WIDTH, device='cuda')), 'WRONG'
print(f'Thread-per-elem Triton: correct')
print(f'{GRID_T}x{GRID_T} programs, each handling a {BLOCK_SIZE}x{BLOCK_SIZE} tile')

#### Benchmark

In [ ]:
RUNS = 10

def bench_cuda(fn, grid, block, args):
    fn(grid, block, args)  # warmup
    cp.cuda.stream.get_current_stream().synchronize()
    t0 = time.time()
    for _ in range(RUNS):
        fn(grid, block, args)
    cp.cuda.stream.get_current_stream().synchronize()
    return (time.time() - t0) / RUNS * 1000

def bench_triton(fn, grid, *args, **kwargs):
    fn[grid](*args, **kwargs)  # warmup
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(RUNS):
        fn[grid](*args, **kwargs)
    torch.cuda.synchronize()
    return (time.time() - t0) / RUNS * 1000

P_out = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)

t_single_cuda    = bench_cuda(matmul_single_cuda,   (1,1),        (1,1),        (M_cp.ravel(), N_cp.ravel(), P_out.ravel(), np.int32(WIDTH)))
t_parallel_cuda  = bench_cuda(matmul_parallel_cuda, (GRID,GRID),  (BLOCK,BLOCK),(M_cp.ravel(), N_cp.ravel(), P_out.ravel(), np.int32(WIDTH)))

P_out_t  = torch.zeros((WIDTH, WIDTH), device='cuda', dtype=torch.float32)
t_single_triton   = bench_triton(matmul_single_triton,   (1,),          M_t, N_t, P_out_t, WIDTH)
t_parallel_triton = bench_triton(matmul_parallel_triton, (GRID_T,GRID_T), M_t, N_t, P_out_t, WIDTH, BLOCK_SIZE=BLOCK_SIZE)

print(f'Single-thread    CUDA:    {t_single_cuda:.2f} ms')
print(f'Single-thread    Triton:  {t_single_triton:.2f} ms')
print(f'Thread-per-elem  CUDA:    {t_parallel_cuda:.2f} ms')
print(f'Thread-per-elem  Triton:  {t_parallel_triton:.2f} ms')
print(f'\nParallel speedup over single-thread (CUDA):   {t_single_cuda/t_parallel_cuda:.1f}x')
print(f'Parallel speedup over single-thread (Triton): {t_single_triton/t_parallel_triton:.1f}x')